In [ ]:
# --- ENVIRONMENT SETUP: Environment-Aware Paths ---
import sys, os
from pathlib import Path

try:
    # Standard Colab setup
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/deep-learning')
except ImportError:
    # Local fallback (assumes running from explorations/ or experiments/)
    cur = Path().resolve()
    REPO_ROOT = cur.parent if cur.name in ('explorations', 'experiments') else cur.parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config.paths import GOLD, BRONZE, SILVER, DATA_LAKE, MODELS, PRETRAINED, TRAINED, OPS, MLFLOW_TRACKING_URI, REPOS


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone --depth 1 -q https://github.com/khangnghiem/deep-learning.git /content/deep-learning
import os, sys, glob, json
sys.path.insert(0, '/content/deep-learning')

EXPERIMENT = '018_polyp_mobilesam'
REPO = 'deep-learning'

print('=' * 60)
print(f'🧪 E2E TEST: {EXPERIMENT}')
print('=' * 60)

In [ ]:
# ✅ 1/5 — Verify data exists
data_path = str(GOLD / '016_polyp_fast_diag_dataset')
assert os.path.exists(data_path), f'❌ Data not found: {data_path}'
items = os.listdir(data_path)
print(f'✅ [1/5] Data verified: {len(items)} items at {data_path}')

In [ ]:
# ✅ 2/5 — Verify experiment files
exp_path = f'/content/{REPO}/experiments/{EXPERIMENT}'
required = ['config.yaml', 'train.py', '018_polyp_mobilesam_train.ipynb', 'README.md']
for f in required:
    assert os.path.exists(os.path.join(exp_path, f)), f'❌ Missing: {f}'
    print(f'  📄 {f}')
print(f'✅ [2/5] All experiment files present')

In [ ]:
# ✅ 3/5 — Verify trained model exists
model_patterns = [
    fstr(TRAINED / '{EXPERIMENT}/**/*.pt'),
]
found = []
for p in model_patterns:
    found.extend(glob.glob(p, recursive=True))
assert len(found) > 0, '❌ No trained model found'
for m in found[:5]:
    size_mb = os.path.getsize(m) / 1024 / 1024
    print(f'  📦 {os.path.basename(m)} ({size_mb:.1f} MB)')
print(f'✅ [3/5] Trained model: {len(found)} file(s)')

In [ ]:
# ✅ 4/5 — Verify MLflow run
import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
runs = mlflow.search_runs(experiment_names=['polyp-mobilesam'], max_results=5)
assert len(runs) > 0, f'❌ No MLflow runs for polyp-mobilesam'
best = runs.iloc[0]
for col in runs.columns:
    if col.startswith('metrics.'):
        val = best[col]
        if val == val:
            print(f'  📊 {col.replace("metrics.", "")}: {val}')
print(f'✅ [4/5] MLflow run: {best["run_id"][:8]}...')

In [ ]:
# ✅ 5/5 — Verify detector inference + MobileSAM
from ultralytics import YOLO, SAM
import torch

det_pt = glob.glob(fstr(TRAINED / '{EXPERIMENT}/**/detector_best.pt'), recursive=True)
assert len(det_pt) > 0, '❌ detector_best.pt not found'

detector = YOLO(det_pt[0])
val_imgs = glob.glob(str(GOLD / '016_polyp_fast_diag_dataset/val/images/*.*'))
assert len(val_imgs) > 0, '❌ No val images'

det_results = detector.predict(val_imgs[0], conf=0.25, verbose=False)
n_boxes = len(det_results[0].boxes) if len(det_results) > 0 else 0
print(f'  Detected {n_boxes} polyp box(es)')

# Quick SAM forward pass
sam = SAM('mobile_sam.pt')
if n_boxes > 0:
    boxes = det_results[0].boxes.xyxy.cpu().numpy()
    sam_results = sam.predict(val_imgs[0], bboxes=boxes[:1], verbose=False)
    print(f'  SAM produced {len(sam_results[0].masks.data)} mask(s)')
print(f'✅ [5/5] Inference test passed')

In [ ]:
print('\n' + '=' * 60)
print('🏁 E2E TEST COMPLETE')
print('=' * 60)
checks = [
    'Data exists and is non-empty',
    'Experiment files are complete',
    'Trained model checkpoint exists',
    'MLflow tracking has logged runs',
    'Detector + MobileSAM inference works',
]
for i, c in enumerate(checks, 1):
    print(f'  ✅ {i}. {c}')
print(f'\n🎉 ALL {len(checks)} CHECKS PASSED')
print('=' * 60)

In [ ]:
from google.colab import runtime
runtime.unassign()
